Start time of execution

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymongo
import yaml
from time import time

from utils import sliding_window_pd, filter_instances
from utils_visual import (
    plot_filtered_vs_raw_signal,
    plot_pairwise_scatter,
    plot_feature_boxplots_by_class,
    plot_scatter_pca,
    plot_window_counts_by_gesture_user,
)
from utils_features import extract_all_candidates

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

**Load Configuation**

In [ ]:
config_path = os.path.join(os.getcwd(), "config.yml")
with open(config_path, "r") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

In [ ]:
time_start = time()


In [ ]:
client = pymongo.MongoClient(config["client"])
db = client[config["db"]]
collection = db[config["col"]]


In [ ]:
found_keys = collection.distinct("_id")
print("Total number of documents in the collection:", len(found_keys))

In [ ]:
documents = collection.find({"_id": {"$in": found_keys}})
rows = []
for doc in documents:
    n_samples = len(doc["data"]["acc_x"])
    for i in range(n_samples):
        rows.append({
            "acc_x": float(doc["data"]["acc_x"][i]),
            "acc_y": float(doc["data"]["acc_y"][i]),
            "acc_z": float(doc["data"]["acc_z"][i]),
            "gyr_x": float(doc["data"]["gyr_x"][i]),
            "gyr_y": float(doc["data"]["gyr_y"][i]),
            "gyr_z": float(doc["data"]["gyr_z"][i]),
            "gesture_id": doc["gesture_id"],

            "user": doc["user"],
        })

dataframe = pd.DataFrame(rows)
# Should be float/int
print(dataframe[["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z", "gesture_id","user"]].dtypes)
print(dataframe[:5])

## Data Exploration
This section will explore the dataset to understand its structure, identify any missing values, and analyze the distribution of the target variable.

### Step 1: Barplot of each recording

In [ ]:
dict = {}
for key,value in dataframe.groupby(["gesture_id", "user"]):
    new_key = f"{key[0]}_{key[1]}"
    dict[new_key] = len(value) / 100 # sampling rate is 100Hz, so we divide by 100 to get the count in seconds
plt.figure(figsize=(12, 6))
sns.barplot(x=list(dict.keys()), y=list(dict.values()))
plt.xticks(rotation=90)
plt.xlabel("Gesture ID and User")
plt.ylabel("Count (in seconds)")
plt.title("Count of Each Gesture ID and User")
plt.show()


### Step 2: Filter Signals
We will apply a low-pass Butterworth filter to the raw signals to remove high-frequency noise and retain the relevant information for our classification task.

In [ ]:
list_of_signals = [value for key,value in dataframe.groupby(["gesture_id","user"])]
filtered_list_of_signals = filter_instances(
        instances_list = list_of_signals,
        order=config["filter"]["order"],
        filter_type=config["filter"]["type"],
        wn = config["filter"]["wn"]
    )

In [ ]:
plot_filtered_vs_raw_signal(
    raw_signals = list_of_signals,
    filtered_signals = filtered_list_of_signals,
    idx = 1,
    n_start = 0,
    n_end = 100
)

### Step 3: Windowing
We will segment the filtered signals into fixed-size windows. Each window will be treated as a separate instance for feature extraction and classification. Each window has distinct user and activity labels. (for example window 1 has user 1 and "swipe up", window 2 has user 1 and "swipe down", etc.)

In [ ]:
# Windowing
windows = []
for signal in filtered_list_of_signals:
    temp_windows = sliding_window_pd(
        df=signal,
        ws=config["sliding_window"]["ws"],
        overlap=config["sliding_window"]["overlap"],
        w_type=config["sliding_window"]["w_type"],
    )
    for window in temp_windows:
        windows.append(window)

print(f"Total number of windows: {len(windows)}")
print(f"Shape of first window: {windows[0].shape}")

### Windowing Barplot
Windowing Barploting in respect to class distribution and user distribution. That helps us to identify if a user and a specific activity is overrepresented in the dataset.


In [ ]:
plot_window_counts_by_gesture_user(windows)

## Train/test split
This procedure happens before feature extraction so we can evaluate the generalization of our features to unseen users.
Also, the split happens based on users, not windows, to avoid data leakage.

In [ ]:
# Train/Test Split by User
train_users = ["01", "02"]
test_users = ["03"]

train_windows = [w for w in windows if w["user"].iloc[0] in train_users]
test_windows = [w for w in windows if w["user"].iloc[0] in test_users]

print(f"Number of windows in training set: {len(train_windows)}")
print(f"Number of windows in testing set: {len(test_windows)}")

### Data Transformation and Feature Engineering
Total Pipeline of Feature Engineering and Data Transformations steps:
1. For each window we will compute exhaustive set of features (statistical, time-domain, frequency-domain, etc.) (function `extract_all_candidates`).
2. For each train window we will aply ANOVA test to select the most relevant features (parameter `k` of `SelectKBest`).
3. Drawing correlation heatmap of the selected features to identify any multicollinearity issues and understand the relationships between features.
4. We will draw pairwise scatter plots of the selected features to visualize their distribution and potential separability between classes. Also, verifies correlation heatmap results.
5. Based on the correlation heatmap and pairwise scatter plots, we will identify and remove any highly correlated features to reduce redundancy and improve model performance.
6. We will apply standard scaling to the selected features to ensure that they are on the same scale, which is important for many machine learning algorithms.
7. We will apply PCA to the scaled features to reduce dimensionality and visualize the data in a 2D space. This can help us to understand the underlying structure of the data and identify any potential clusters or patterns.

In [ ]:

features_df = pd.DataFrame([extract_all_candidates(window) for window in train_windows])
features_df_test = pd.DataFrame([extract_all_candidates(window) for window in test_windows])
print("Extracted features shape:", features_df.shape)
print("First few rows of the features dataframe:")
print(features_df.head())

In [ ]:
X_train = features_df.drop(["user","gesture_id"], axis=1)
y_train = features_df["gesture_id"]
selector = SelectKBest(f_classif, k=12)
selector.fit(X_train, y_train)
scores   = pd.Series(selector.scores_, index=X_train.columns).sort_values(ascending=False)
selected = X_train.columns[selector.get_support()].tolist()
print("score: ")
print(scores.head(12))
print("ANOVA F-scores (top 12):")
print(selected)

In [ ]:
corr_matrix = features_df[selected].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix of Selected Features", fontsize=16, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Add gesture labels to features dataframe for visualization
features_df_with_labels = features_df.copy()
if "gesture_id" not in features_df_with_labels.columns and len(y_train) == len(features_df):
    features_df_with_labels["gesture_id"] = y_train.values

plot_pairwise_scatter(
    df=features_df_with_labels,
    features=selected,
    label_col="gesture_id",
    display_type="grid"
)

### Drop Correlated Features
We are going to drop features that are highly correlated with each other, as they do not provide additional information to the model and can lead to overfitting. We will use a correlation threshold of 0.9, meaning that if two features have a correlation coefficient greater than 0.9, we will drop one of them.

In [ ]:
# Drop highly correlated features
# Keep: acc_y_rms (highest ANOVA F-score)
# Drop: acc_y_mean, acc_mag_mean

columns_to_keep = [col for col in selected if col not in ["acc_y_mean", "acc_mag_mean"]]
df_boxplot_features = features_df_with_labels[columns_to_keep + ["gesture_id"]]



In [ ]:
# Feature boxplots grouped by gesture class
plot_feature_boxplots_by_class(
    df=df_boxplot_features,
    features=columns_to_keep,  # Use the reduced feature set
    label_col="gesture_id",
    figsize_per_plot=(6, 5)
)

### PCA Scatter Plot
To visualize the distribution of the data in a lower-dimensional space, we will use Principal Component Analysis (PCA). PCA is a dimensionality reduction technique that transforms the data into a new coordinate system, where the greatest variance of the data is captured in the first few principal components. We will create a scatter plot of the first two principal components to see how the different classes are distributed in this reduced space.

In [ ]:
# Apply StandardScaler to the selected features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(features_df[columns_to_keep])
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=columns_to_keep)


In [ ]:
from utils_visual import plot_scatter_pca
X_pca = PCA(n_components=3)
X_train_pca = X_pca.fit_transform(X_train_scaled_df)
X_train_pca_df = pd.DataFrame(X_train_pca, columns=["PC1", "PC2", "PC3"])
X_train_pca_df["encoded_gesture_id"] = features_df["gesture_id"].astype("category").cat.codes
plot_scatter_pca(
    df=X_train_pca_df,
    c_name="encoded_gesture_id"
)

### Model Evaluation
After feature engineering, we will evaluate the performance of our model using the selected features. We will use a classification algorithm (e.g., Random Forest, SVM) and evaluate it using metrics such as accuracy, precision, recall, and F1-score. We will also use cross-validation to ensure that our results are robust and not due to random chance.

In [ ]:
X_train_model = features_df[columns_to_keep]
y_train = features_df["gesture_id"]
X_test_model = features_df_test[columns_to_keep]

In [ ]:
X_test = features_df_test[columns_to_keep]
y_test = features_df_test["gesture_id"]
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        random_state=42
    ))
])
svm_pipeline.fit(X_train_model, y_train)
y_pred_svm = svm_pipeline.predict(X_test_model)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_model, y_train)
y_pred_rf = rf_model.predict(X_test_model)

In [ ]:
print("SVM Classification Report:")
print(classification_report(y_test, y_pred_svm))
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
time_end = time()
elapsed_time = time_end - time_start
print(f"Total execution time: {elapsed_time:.2f} seconds")